
# Mini Lakehouse Demo – *Online Retail Sales*  

*(Databricks Free Edition ✕ Cloudflare R2 Object Storage)*  

- Zero-cost setup: Databricks Free Edition + Cloudflare R2 free tier
- Governed storage: Unity Catalog external volume on R2, secured with a storage credential
- Fast ingest: one wildcard Spark read loads all CSV snapshots (~1 M rows in < 5 s)

- Data quality: explicit schema, integer / decimal casting, multi-format date normalisation in a single expression
- Bronze Delta table: main_catalog.online_retail.sales_raw with ACID, time travel and automatic schema merge

- Incremental ready: same path supports COPY INTO so only new files are loaded on future runs
- Lineage & audit: every read/write captured by Unity Catalog History and visible in Catalog Explorer
- Cloud-agnostic paths: code uses /Volumes/... instead of raw bucket URLs for easy migration

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, types as T

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main_catalog.online_retail;

## Bronze layer data preparation

In [0]:
VOLUME_PATH = "/Volumes/main_catalog/online_retail/raw/"  

schema = T.StructType([                       # explicit schema 
    T.StructField("Invoice",      T.StringType()),
    T.StructField("StockCode",    T.StringType()),
    T.StructField("Description",  T.StringType()),
    T.StructField("Quantity",     T.IntegerType()),        
    T.StructField("InvoiceDate",  T.StringType()),         
    T.StructField("UnitPrice",    T.DecimalType(10,2)),    
    T.StructField("CustomerID",   T.DecimalType(10,0)),
    T.StructField("Country",      T.StringType())
])

#Create df_raw to merge all raw data

df_raw = (spark.read
              .option("header", "true")
              .option("sep", ";")
              .schema(schema)
              .csv(f"{VOLUME_PATH}online_retail_utf_8_*.csv"))

# Correcting dates

df_fixed = (
    df_raw
      .withColumn(
          "InvoiceDate",
          F.coalesce(
              F.to_timestamp("InvoiceDate", "yyyy.MM.dd H:mm"),
              F.to_timestamp("InvoiceDate", "yyyy.MM.dd HH:mm"),
              F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss")
          )
      )
)

# Store as delta table and register as table
TARGET_TABLE = "main_catalog.online_retail.sales_raw"      
(df_fixed
   .write
   .format("delta")
   .mode("overwrite")         
   .option("overwriteSchema", "true")
   .saveAsTable(TARGET_TABLE))

print("Sales_raw ready:", spark.table(TARGET_TABLE).count(), "rows")

## Silver layer data preparation

- Adding year / month col 
- data cleaning if unitprice is 0 than drop it. There is no free items in this store

In [0]:
%sql
-- silver layer ─ cleaned & partitioned
CREATE OR REPLACE TABLE main_catalog.online_retail.sales_silver
USING DELTA
PARTITIONED BY (invoice_year, invoice_month)
AS
WITH src AS (
  SELECT *,                    -- bring everything
         to_timestamp(InvoiceDate, 'yyyy.MM.dd H:mm') AS ts
  FROM   main_catalog.online_retail.sales_raw
  WHERE  UnitPrice > 0
)
SELECT
    Invoice,
    StockCode,
    COALESCE(Description, 'Unknown')            AS Description,
    Quantity,
    ts                                          AS InvoiceTS,
    YEAR(ts)                                    AS invoice_year,
    MONTH(ts)                                   AS invoice_month,
    CAST(UnitPrice AS DECIMAL(10,2))                AS Price,
    CustomerID,
    Country
FROM src;

In [0]:
%sql 
WITH excepted_records AS (
SELECT *
FROM workspace.online_retail.sales_raw sales_raw
EXCEPT ALL
SELECT 
       sales_silver.Invoice,
       sales_silver.StockCode,
       sales_silver.Description,
       sales_silver.Quantity,
       sales_silver.InvoiceTS as InvoiceDate,
       sales_silver.Price,
       sales_silver.CustomerID as customer_id,
       sales_silver.Country
FROM workspace.online_retail.sales_silver sales_silver
)
SELECT * 
FROM excepted_records r --- 62225 records is null customer_id or price is 0
WHERE r.customer_id IS NOT NULL

##  Create a hierarchie level table for articles

In [0]:
silver_df = spark.table("main_catalog.online_retail.sales_silver")

# build the “articles” dimension  (distinct + null-handling)
articles_df = (silver_df
               .select(
                   "StockCode",
                   F.coalesce("Description", F.lit("Unknown")).alias("Description")
               )
               .dropDuplicates())

# overwrite / create the Delta table inside Unity Catalog
(articles_df
   .write
   .format("delta")
   .mode("overwrite")           
   .option("overwriteSchema", "true")
   .saveAsTable("main_catalog.online_retail.sales_articles"))

In [0]:
%sql
SELECT 
    a.StockCode,
    a.Description
FROM main_catalog.online_retail.sales_articles a  LIMIT 10;